In [ ]:
"""
Wokwi Real-Time Data Fetcher
Polls the Wokwi simulation JSON API and displays live sensor/pin data.
"""

import requests
import time
import json
from datetime import datetime

# ── Configuration ────────────────────────────────────────────────────────────
WOKWI_URL     = "http://10.10.0.2/json"
POLL_INTERVAL = 1.0          # seconds between each fetch
TIMEOUT       = 5            # request timeout in seconds
MAX_RETRIES   = 3            # consecutive failures before warning
LOG_TO_FILE   = True         # set False to disable file logging
LOG_FILE      = "wokwi_log.json"
# ─────────────────────────────────────────────────────────────────────────────


def fetch_simulation_data(url: str, timeout: int) -> dict | None:
    """Fetch JSON data from the Wokwi simulation endpoint."""
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.ConnectionError:
        print(f"  [ERROR] Cannot connect to {url}. Is the simulation running?")
    except requests.exceptions.Timeout:
        print(f"  [ERROR] Request timed out after {timeout}s.")
    except requests.exceptions.HTTPError as e:
        print(f"  [ERROR] HTTP error: {e}")
    except json.JSONDecodeError:
        print(f"  [ERROR] Response is not valid JSON.")
    return None


def display_data(data: dict, timestamp: str) -> None:
    """Pretty-print the simulation data to the console."""
    print(f"\n{'─' * 55}")
    print(f"  Timestamp : {timestamp}")
    print(f"{'─' * 55}")

    # ── Top-level simulation metadata ────────────────────────────────────────
    sim_keys = ["time", "millis", "status", "running"]
    for key in sim_keys:
        if key in data:
            print(f"  {key:<14}: {data[key]}")

    # ── Components / pins / sensors ──────────────────────────────────────────
    for section in ["pins", "components", "sensors", "analog", "digital"]:
        if section in data and data[section]:
            print(f"\n  [{section.upper()}]")
            items = data[section]
            if isinstance(items, dict):
                for k, v in items.items():
                    print(f"    {k:<20}: {v}")
            elif isinstance(items, list):
                for item in items:
                    if isinstance(item, dict):
                        line = "  ".join(f"{k}={v}" for k, v in item.items())
                        print(f"    {line}")
                    else:
                        print(f"    {item}")

    # ── Any other keys not handled above ─────────────────────────────────────
    handled = {"time", "millis", "status", "running",
               "pins", "components", "sensors", "analog", "digital"}
    extras = {k: v for k, v in data.items() if k not in handled}
    if extras:
        print(f"\n  [OTHER]")
        for k, v in extras.items():
            print(f"    {k:<20}: {v}")

    print(f"{'─' * 55}")


def log_to_file(data: dict, timestamp: str, filepath: str) -> None:
    """Append a timestamped record to a JSON-lines log file."""
    record = {"timestamp": timestamp, "data": data}
    with open(filepath, "a") as f:
        f.write(json.dumps(record) + "\n")


def main() -> None:
    print("=" * 55)
    print("  Wokwi Real-Time Monitor")
    print(f"  Endpoint : {WOKWI_URL}")
    print(f"  Interval : {POLL_INTERVAL}s")
    if LOG_TO_FILE:
        print(f"  Log file : {LOG_FILE}")
    print("  Press Ctrl+C to stop.")
    print("=" * 55)

    consecutive_failures = 0

    while True:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        data = fetch_simulation_data(WOKWI_URL, TIMEOUT)

        if data is not None:
            consecutive_failures = 0
            display_data(data, timestamp)
            if LOG_TO_FILE:
                log_to_file(data, timestamp, LOG_FILE)
        else:
            consecutive_failures += 1
            if consecutive_failures >= MAX_RETRIES:
                print(f"\n  ⚠  {consecutive_failures} consecutive failures. "
                      "Check that the Wokwi simulation is active.\n")

        try:
            time.sleep(POLL_INTERVAL)
        except KeyboardInterrupt:
            print("\n\nMonitoring stopped by user. Goodbye!")
            break


if __name__ == "__main__":
    main()

In [ ]:
import time
import math

def get_vitals():
    # Mirrors exactly what your Wokwi is printing
    # Slowly rises to critical values — same as simulation
    t = time.time()
    return {
        "hr_pulse":    int(60 + (t % 60)),           # 60 → 120 BPM
        "hr_max30100": int(72 + (t / 5) % 50),       # 72 → 122 BPM
        "spo2":        round(99 - (t / 10) % 9, 1),  # 99 → 91%
        "body_temp":   round(36.5 + (t / 30) % 2, 1),# 36.5 → 38.5°C
        "room_temp":   22.0,
        "humidity":    60.0
    }

In [ ]:
import json
import requests

OPENROUTER_API_KEY = "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"

def get_ai_response(health_data: dict) -> str:
    prompt = f"""You are a medical assistant. Analyze the following patient health sensor data and provide a brief clinical assessment:

{json.dumps(health_data, indent=2)}

Comment on each vital sign, whether it is normal or abnormal, and summarize the overall health status."""

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta-llama/llama-3-8b-instruct",  # or llama-3-70b-instruct
            "max_tokens": 300,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    result = response.json()
    return result["choices"][0]["message"]["content"]


# Example usage with your sensor data
health_data = {
    "hr_pulse": "93 BPM",
    "hr_max30100": "112 BPM",
    "spo2": "98.7 %",
    "body_temp": "37.6 °C",
    "room_temp": "22.0 °C",
    "humidity": "60.0 %",
    "alert": "CRITICAL: Tachycardia!"
}

result = get_ai_response(health_data)
print(result)

In [ ]:
import json
import requests

OPENROUTER_API_KEY = "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"

def load_health_data(filepath: str) -> dict:
    with open(filepath, "r") as f:
        return json.load(f)

def get_ai_response(health_data: dict) -> str:
    prompt = f"""You are a medical assistant. Analyze the following patient health sensor data and provide a brief clinical assessment:

{json.dumps(health_data, indent=2)}

Comment on each vital sign, whether it is normal or abnormal, and summarize the overall health status."""

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta-llama/llama-3-8b-instruct",
            "max_tokens": 300,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    result = response.json()
    return result["choices"][0]["message"]["content"]


# ---- Main ----
if __name__ == "__main__":
    filepath = "health_data.json"          # Change to your JSON file path
    health_data = load_health_data(filepath)
    print(f"Loaded data: {json.dumps(health_data, indent=2)}\n")

    result = get_ai_response(health_data)
    print("AI Assessment:")
    print(result)

In [ ]:
raw = """
Pulse Sensor BPM : 0
MAX30100 BPM     : 73.00
SpO2             : 99.00%
Body Temperature : 22.00°C
Room Temperature : 2.10°C
Room Humidity    : 15.40%
"""

for line in raw.strip().splitlines():
    if ":" in line:
        key, val = line.split(":", 1)
        print(f"{key.strip()} → {val.strip()}")

In [ ]:
pip install requests

In [ ]:
pip install watchdog

In [ ]:
"""
Wokwi Real-Time Data Fetcher
Uses simulated data mirroring Wokwi ESP32 output
"""

import time
import json
from datetime import datetime

# ── Configuration ────────────────────────────────────────────────────────────
POLL_INTERVAL = 1.0
LOG_TO_FILE   = True
LOG_FILE      = "wokwi_log.json"
# ─────────────────────────────────────────────────────────────────────────────

def get_simulated_vitals() -> dict:
    """
    Mirrors exactly what the Wokwi ESP32 outputs via serial.
    Values slowly rise to critical — same as simulation behaviour.
    """
    t = time.time()
    return {
        "hr_pulse":    int(60 + (t % 60)),
        "hr_max30100": int(72 + (t / 5) % 50),
        "spo2":        round(99 - (t / 10) % 9, 1),
        "body_temp":   round(36.5 + (t / 30) % 2, 1),
        "room_temp":   22.0,
        "humidity":    60.0
    }

def display_data(data: dict, timestamp: str) -> None:
    print(f"\n{'─' * 55}")
    print(f"  Timestamp    : {timestamp}")
    print(f"{'─' * 55}")
    print(f"  hr_pulse     : {data['hr_pulse']} BPM")
    print(f"  hr_max30100  : {data['hr_max30100']} BPM")
    print(f"  spo2         : {data['spo2']} %")
    print(f"  body_temp    : {data['body_temp']} °C")
    print(f"  room_temp    : {data['room_temp']} °C")
    print(f"  humidity     : {data['humidity']} %")

    # Danger alerts
    if data['spo2'] < 94:
        print(f"  🔴 CRITICAL  : SpO2 low!")
    if data['hr_max30100'] > 110:
        print(f"  🔴 CRITICAL  : Tachycardia!")
    if data['body_temp'] > 38.3:
        print(f"  🟡 WARNING   : Fever!")
    if data['spo2'] < 94 and data['hr_max30100'] > 110:
        print(f"  🚨 RISK      : SEPSIS/PE — Escalate!")

    print(f"{'─' * 55}")

def log_to_file(data: dict, timestamp: str, filepath: str) -> None:
    record = {"timestamp": timestamp, "data": data}
    with open(filepath, "a") as f:
        f.write(json.dumps(record) + "\n")

def get_vitals() -> dict:
    """Main function used by serial_reader and main.py"""
    return get_simulated_vitals()

def main() -> None:
    print("=" * 55)
    print("  Wokwi Real-Time Monitor")
    print("  Source   : Simulated (mirrors Wokwi ESP32 output)")
    print(f"  Interval : {POLL_INTERVAL}s")
    if LOG_TO_FILE:
        print(f"  Log file : {LOG_FILE}")
    print("  Press Ctrl+C to stop.")
    print("=" * 55)

    while True:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        data = get_simulated_vitals()

        display_data(data, timestamp)

        if LOG_TO_FILE:
            log_to_file(data, timestamp, LOG_FILE)

        try:
            time.sleep(POLL_INTERVAL)
        except KeyboardInterrupt:
            print("\n\nMonitoring stopped. Goodbye!")
            break

if __name__ == "__main__":
    main()

In [ ]:
"""
Unified Wokwi Real-Time Monitor + AI Assessment
Serial reader feeds directly into AI pipeline
"""

import time
import json
import requests
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = "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"
POLL_INTERVAL      = 5.0    # seconds between each reading + AI call
LOG_TO_FILE        = True
LOG_FILE           = "wokwi_log.json"

# AI call cooldown — only call AI if vitals have actually changed
AI_CHANGE_THRESHOLD = {
    "hr_pulse":    2,     # BPM difference to trigger AI
    "hr_max30100": 2,
    "spo2":        0.5,
    "body_temp":   0.1,
}
# ─────────────────────────────────────────────────────────────────────────────

def get_simulated_vitals() -> dict:
    """Mirrors exactly what the Wokwi ESP32 outputs via serial."""
    t = time.time()
    return {
        "hr_pulse":    int(60 + (t % 60)),
        "hr_max30100": int(72 + (t / 5) % 50),
        "spo2":        round(99 - (t / 10) % 9, 1),
        "body_temp":   round(36.5 + (t / 30) % 2, 1),
        "room_temp":   22.0,
        "humidity":    60.0
    }

def get_alerts(data: dict) -> list:
    """Returns list of alert strings based on vitals."""
    alerts = []
    if data["spo2"] < 94:
        alerts.append("🔴 CRITICAL  : SpO2 low!")
    if data["hr_max30100"] > 110:
        alerts.append("🔴 CRITICAL  : Tachycardia!")
    if data["body_temp"] > 38.3:
        alerts.append("🟡 WARNING   : Fever!")
    if data["spo2"] < 94 and data["hr_max30100"] > 110:
        alerts.append("🚨 RISK      : SEPSIS/PE — Escalate!")
    return alerts

def display_data(data: dict, timestamp: str) -> None:
    print(f"\n{'─' * 55}")
    print(f"  Timestamp    : {timestamp}")
    print(f"{'─' * 55}")
    print(f"  hr_pulse     : {data['hr_pulse']} BPM")
    print(f"  hr_max30100  : {data['hr_max30100']} BPM")
    print(f"  spo2         : {data['spo2']} %")
    print(f"  body_temp    : {data['body_temp']} °C")
    print(f"  room_temp    : {data['room_temp']} °C")
    print(f"  humidity     : {data['humidity']} %")
    for alert in get_alerts(data):
        print(f"  {alert}")
    print(f"{'─' * 55}")

def log_to_file(data: dict, timestamp: str, ai_response: str) -> None:
    record = {"timestamp": timestamp, "data": data, "ai_assessment": ai_response}
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")

def has_significant_change(current: dict, previous: dict) -> bool:
    """Returns True if any vital changed beyond threshold."""
    if previous is None:
        return True
    for key, threshold in AI_CHANGE_THRESHOLD.items():
        if abs(current.get(key, 0) - previous.get(key, 0)) >= threshold:
            return True
    return False

def get_ai_response(health_data: dict, alerts: list) -> str:
    alert_text = "\n".join(alerts) if alerts else "No critical alerts."
    prompt = f"""You are a medical assistant. Analyze the following real-time patient vitals from a hardware sensor:

{json.dumps(health_data, indent=2)}

Active Alerts:
{alert_text}

Briefly assess each vital sign (normal/abnormal), explain any alerts, and summarize the patient's current health status."""

    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "meta-llama/llama-3-8b-instruct",
                "max_tokens": 300,
                "messages": [{"role": "user", "content": prompt}]
            },
            timeout=15
        )
        result = response.json()
        return result["choices"][0]["message"]["content"]
    except requests.exceptions.Timeout:
        return "[AI ERROR] Request timed out."
    except Exception as e:
        return f"[AI ERROR] {str(e)}"

# ── Main Loop ─────────────────────────────────────────────────────────────────
def main() -> None:
    print("=" * 55)
    print("  Wokwi Real-Time Monitor + AI Assessment")
    print(f"  Poll Interval : {POLL_INTERVAL}s")
    print(f"  Log File      : {LOG_FILE}")
    print("  Press Ctrl+C to stop.")
    print("=" * 55)

    previous_data = None

    while True:
        try:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            data      = get_simulated_vitals()
            alerts    = get_alerts(data)

            # Always display vitals
            display_data(data, timestamp)

            # Only call AI if vitals changed significantly
            if has_significant_change(data, previous_data):
                print("\n  [AI] Vitals changed — getting assessment...")
                ai_result = get_ai_response(data, alerts)
                print(f"\n  AI Assessment:\n  {ai_result.replace(chr(10), chr(10) + '  ')}")
                print(f"{'─' * 55}")

                if LOG_TO_FILE:
                    log_to_file(data, timestamp, ai_result)

                previous_data = data.copy()
            else:
                print("  [AI] No significant change — skipping AI call.")

            time.sleep(POLL_INTERVAL)

        except KeyboardInterrupt:
            print("\n\n  Monitoring stopped. Goodbye!")
            break

if __name__ == "__main__":
    main()
    

In [ ]:
import time
import json
from datetime import datetime

POLL_INTERVAL = 1.0
LOG_TO_FILE   = True
LOG_FILE      = "wokwi_log.json"

_start_time = time.time()

def get_vitals() -> dict:
    """
    3 phases — 5 minutes each:
    Phase 0 (0-5min)   → NORMAL
    Phase 1 (5-10min)  → WARNING
    Phase 2 (10-15min) → CRITICAL
    Then repeats
    """
    elapsed = time.time() - _start_time
    phase   = int(elapsed / 300) % 3  # 300 seconds = 5 minutes

    if phase == 0:
        vitals = {
            "hr_pulse":    72,
            "hr_max30100": 75,
            "spo2":        98.0,
            "body_temp":   36.6,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "NORMAL"
        }
    elif phase == 1:
        vitals = {
            "hr_pulse":    95,
            "hr_max30100": 98,
            "spo2":        95.0,
            "body_temp":   37.8,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "WARNING"
        }
    else:
        vitals = {
            "hr_pulse":    118,
            "hr_max30100": 122,
            "spo2":        91.0,
            "body_temp":   38.7,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "CRITICAL"
        }

    icon = "✅" if phase == 0 else "🟡" if phase == 1 else "🚨"
    mins_remaining = int((300 - (elapsed % 300)) / 60)
    secs_remaining = int((300 - (elapsed % 300)) % 60)

    print(f"[VITALS] {icon} {vitals['phase']} | "
          f"HR={vitals['hr_max30100']} | "
          f"SpO2={vitals['spo2']}% | "
          f"Temp={vitals['body_temp']}°C | "
          f"Next phase in {mins_remaining}m {secs_remaining}s")
    return vitals


def display_data(data: dict, timestamp: str) -> None:
    phase = data.get("phase", "UNKNOWN")
    icon  = "✅" if phase == "NORMAL" else "🟡" if phase == "WARNING" else "🚨"

    elapsed = time.time() - _start_time
    mins_remaining = int((300 - (elapsed % 300)) / 60)
    secs_remaining = int((300 - (elapsed % 300)) % 60)

    print(f"\n{'─' * 60}")
    print(f"  {icon} Phase        : {phase}  "
          f"(changes in {mins_remaining}m {secs_remaining}s)")
    print(f"  Timestamp    : {timestamp}")
    print(f"{'─' * 60}")
    print(f"  hr_pulse     : {data['hr_pulse']} BPM")
    print(f"  hr_max30100  : {data['hr_max30100']} BPM")
    print(f"  spo2         : {data['spo2']} %")
    print(f"  body_temp    : {data['body_temp']} °C")
    print(f"  room_temp    : {data['room_temp']} °C")
    print(f"  humidity     : {data['humidity']} %")

    if data['spo2'] < 94:
        print(f"  🔴 CRITICAL  : SpO2 = {data['spo2']}% — Hypoxemia!")
    if data['hr_max30100'] > 110:
        print(f"  🔴 CRITICAL  : HR = {data['hr_max30100']} BPM — Tachycardia!")
    if data['body_temp'] > 38.3:
        print(f"  🟡 WARNING   : Temp = {data['body_temp']}°C — Fever!")
    if data['spo2'] < 94 and data['hr_max30100'] > 110:
        print(f"  🚨 RISK      : SEPSIS/PE — Escalate Immediately!")

    print(f"{'─' * 60}")


def log_to_file(data: dict, timestamp: str, filepath: str) -> None:
    record = {"timestamp": timestamp, "data": data}
    with open(filepath, "a") as f:
        f.write(json.dumps(record) + "\n")


def main() -> None:
    print("=" * 60)
    print("  VITAL Watch — Wokwi Real-Time Monitor")
    print("  Phases (5 minutes each):")
    print("  ✅  0-5min  → NORMAL   (HR=75  SpO2=98%  Temp=36.6°C)")
    print("  🟡  5-10min → WARNING  (HR=98  SpO2=95%  Temp=37.8°C)")
    print("  🚨 10-15min → CRITICAL (HR=122 SpO2=91%  Temp=38.7°C)")
    print("  Then repeats...")
    print(f"  Interval : {POLL_INTERVAL}s")
    print("  Press Ctrl+C to stop.")
    print("=" * 60)

    while True:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        data = get_vitals()
        display_data(data, timestamp)
        if LOG_TO_FILE:
            log_to_file(data, timestamp, LOG_FILE)
        try:
            time.sleep(POLL_INTERVAL)
        except KeyboardInterrupt:
            print("\n\nMonitoring stopped. Goodbye!")
            break


if __name__ == "__main__":
    main()